# M3L3 E19 — Sistema empresarial completo con evaluador automático
### Módulo 3 · Lecture 3 · Sistemas Multiagente Avanzados

**Caso terminado:** sistema empresarial con 4 departamentos especializados, un evaluador automático de calidad y soporte opcional de trazabilidad con Langfuse.

## ¿Qué vas a ver en este ejercicio?
- 4 departamentos corporativos: HR, IT, Finance, Legal.
- Un **evaluador automático** que audita cada respuesta en 4 dimensiones (relevancia, completitud, precisión, claridad).
- Integración opcional con **Langfuse** para trazabilidad de producción.

## Arquitectura del sistema

> **Auto-evaluador:** nodo que recibe la respuesta de un agente especialista y la audita usando el mismo LLM. Produce métricas de calidad sin intervención humana.

```
START
  |
  v
enterprise_router_node
  |       |         |        |        |
  v       v         v        v        v
 hr_agent it_agent fin_agent leg_agent fallback
  |       |         |        |        |
  +-------+---------+--------+--------+
                    |
            evaluator_agent
                    |
                   END
```

| Nodo | Rol |
|---|---|
| `enterprise_router_node` | Clasifica el departamento con LLM |
| `hr_agent` | Políticas de RRHH, vacaciones, contratación |
| `it_agent` | Infraestructura, seguridad, SaaS corporativo |
| `finance_agent` | Presupuestos, gastos, reportes financieros |
| `legal_agent` | Contratos, compliance, NDA, regulaciones |
| `fallback_agent` | Consultas fuera de alcance |
| `evaluator_agent` | Audita la calidad de la respuesta (0–10 por dimensión) |

## Paso 1 — Elegí tu proveedor de LLM

In [ ]:
PROVIDER = "openai"   # ← cambiá esto: "openai" | "gemini" | "claude"

import os
from getpass import getpass

if PROVIDER == "openai":
    !pip install langchain-openai -q
    from langchain_openai import ChatOpenAI
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

elif PROVIDER == "gemini":
    !pip install langchain-google-genai -q
    from langchain_google_genai import ChatGoogleGenerativeAI
    os.environ["GOOGLE_API_KEY"] = getpass("Google API Key: ")
    llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash", temperature=0)

elif PROVIDER == "claude":
    !pip install langchain-anthropic -q
    from langchain_anthropic import ChatAnthropic
    os.environ["ANTHROPIC_API_KEY"] = getpass("Anthropic API Key: ")
    llm = ChatAnthropic(model="claude-haiku-4-5-20251001", temperature=0)

else:
    raise ValueError(f"PROVIDER inválido: {PROVIDER!r}. Opciones: 'openai' | 'gemini' | 'claude'")

print(f"LLM listo → proveedor: {PROVIDER}")

## Paso 2 — Instalar dependencias

In [ ]:
!pip install langgraph -q

from typing import TypedDict
from langgraph.graph import StateGraph, START, END
import json, re

print("LangGraph listo.")

## Paso 3 (opcional) — Langfuse para trazabilidad

> **Langfuse:** plataforma de observabilidad para sistemas LLM. Cada `invoke` puede quedar registrado como un trace con inputs, outputs, latencia y costos. Útil en producción para auditar el comportamiento del sistema.

In [ ]:
LANGFUSE_ENABLED = False   # ← cambiar a True si querés usar Langfuse

langfuse_client = None

if LANGFUSE_ENABLED:
    try:
        !pip install langfuse -q
        from langfuse import Langfuse
        os.environ["LANGFUSE_PUBLIC_KEY"]  = getpass("Langfuse Public Key: ")
        os.environ["LANGFUSE_SECRET_KEY"]  = getpass("Langfuse Secret Key: ")
        os.environ["LANGFUSE_HOST"]        = "https://cloud.langfuse.com"
        langfuse_client = Langfuse()
        print("Langfuse conectado.")
    except Exception as e:
        print(f"Langfuse no disponible: {e}. El sistema funciona igual sin él.")
        LANGFUSE_ENABLED = False
else:
    print("Langfuse deshabilitado. Setear LANGFUSE_ENABLED=True para activar.")

## Sección 1 — Knowledge bases corporativas

In [ ]:
hr_kb = [
    "Vacaciones: 15 días hábiles anuales para empleados con menos de 5 años; 20 días para más de 5 años.",
    "Licencia por maternidad/paternidad: 90 días para madre, 15 días para padre, remunerada al 100%.",
    "Proceso de contratación: 3 etapas (técnica, cultura fit, oferta). Duración estimada: 3-4 semanas.",
    "Beneficios: obra social premium, prepaga, ticket restaurante $5.000/mes, home office 2 días/semana.",
    "Revisión salarial: anual en diciembre, basada en performance review del trimestre anterior.",
    "Capacitación: presupuesto de $200.000/año por empleado para cursos, certificaciones o conferencias.",
    "Política de trabajo remoto: hasta 100% remoto previa aprobación del manager directo.",
]

it_kb = [
    "Acceso a sistemas: solicitar via ticket en el portal IT con aprobación del manager.",
    "VPN corporativa: usar GlobalProtect, reiniciar si hay problemas de conexión y validar MFA.",
    "Notebook corporativa: reposición en 3 años o ante falla mayor. Solicitar via formulario IT.",
    "SaaS aprobados: Google Workspace, Slack, Jira, GitHub, Figma, Zoom, Notion.",
    "Contraseñas: política de 12 caracteres mínimo, rotación cada 90 días, gestor LastPass disponible.",
    "Incidentes de seguridad: reportar inmediatamente a security@empresa.com y al manager.",
    "Backup: datos críticos en Google Drive corporativo con retención de 1 año.",
]

finance_kb = [
    "Gastos: reembolsables con ticket/factura y aprobación del manager hasta $50.000. Más: requiere VP.",
    "Presupuestos: proceso anual en noviembre. Solicitudes de ajuste mid-year via Finance.",
    "Viajes corporativos: vuelos en económica salvo trayectos >8hs. Hotel máximo $200/noche.",
    "Cierre mensual: los 5 primeros días hábiles del mes siguiente. Facturas deben entrar antes.",
    "Centro de costos: cada equipo tiene su propio CC. Validar con Finance antes de imputar.",
    "Tarjeta corporativa: disponible para directores y VPs. Gastos con justificación obligatoria.",
    "Reportes financieros: P&L mensual publicado el día 10 en el portal de Finance.",
]

legal_kb = [
    "NDAs: toda relación con terceros requiere NDA firmado. Plantillas en el portal Legal.",
    "Contratos con proveedores: revisión obligatoria por Legal para montos >$500.000.",
    "Compliance GDPR/LGPD: datos de usuarios EU/BR requieren consentimiento explícito y DPA.",
    "Propiedad intelectual: todo código desarrollado en horario laboral pertenece a la empresa.",
    "Política de conflictos de interés: declarar cualquier relación con competidores o proveedores.",
    "Litigios: cualquier notificación judicial debe escalarse inmediatamente al departamento Legal.",
    "Contratos de empleados: regidos por el Convenio Colectivo de Trabajo del sector tecnológico.",
]

enterprise_kbs = {
    "hr": hr_kb,
    "it": it_kb,
    "finance": finance_kb,
    "legal": legal_kb,
}

print("Knowledge bases corporativas cargadas:", list(enterprise_kbs.keys()))

## Sección 2 — State empresarial

In [ ]:
class EnterpriseState(TypedDict):
    query: str
    department: str        # "hr" | "it" | "finance" | "legal" | "unknown"
    reason: str
    agent_response: str    # respuesta del agente especialista
    eval_score: float      # promedio de las 4 dimensiones (0.0–10.0)
    eval_relevance: float
    eval_completeness: float
    eval_accuracy: float
    eval_clarity: float
    eval_feedback: str     # feedback textual del evaluador

## Sección 3 — Nodos del sistema

> **Evaluador automático:** usa el mismo LLM para auditar la respuesta de otro agente. El prompt pide un JSON estructurado con scores y feedback. Este patrón se llama "LLM-as-a-judge".

In [ ]:
def enterprise_router_node(state: EnterpriseState) -> dict:
    prompt = (
        "Sos el router de un sistema de soporte empresarial corporativo.\n"
        "Clasificá la consulta en exactamente un departamento:\n"
        "- 'hr': recursos humanos, vacaciones, beneficios, contratación, desvinculación\n"
        "- 'it': tecnología, infraestructura, acceso a sistemas, seguridad informática\n"
        "- 'finance': presupuestos, gastos, reembolsos, facturas, reportes financieros\n"
        "- 'legal': contratos, compliance, NDA, propiedad intelectual, litigios\n"
        "- 'unknown': consultas que no corresponden a ningún departamento\n\n"
        "Respondé con JSON: {\"department\": \"...\", \"reason\": \"...\"}\n"
        "Solo department y reason, sin markdown.\n\n"
        f"Consulta: {state['query']}"
    )
    response = llm.invoke(prompt)
    text = response.content.strip()
    text = re.sub(r"```[\w]*\n?", "", text).strip()
    try:
        data = json.loads(text)
        department = data.get("department", "unknown").lower()
        reason = data.get("reason", "")
    except Exception:
        department = "unknown"
        reason = text
    if department not in ("hr", "it", "finance", "legal"):
        department = "unknown"
    return {"department": department, "reason": reason}


print("Router definido.")

In [ ]:
def _enterprise_rag_agent(dept: str, display: str, state: EnterpriseState) -> dict:
    context = "\n".join(enterprise_kbs[dept])
    response = llm.invoke(
        f"Sos el agente de {display} de una empresa tecnológica.\n"
        "Respondé la consulta del empleado usando únicamente el contexto provisto.\n"
        "Si la información no está en el contexto, indicalo claramente sin inventar.\n\n"
        f"Política y procedimientos de {display}:\n{context}\n\n"
        f"Consulta del empleado: {state['query']}\n\n"
        "Respondé de forma clara, precisa y profesional en español."
    )
    return {"agent_response": response.content.strip()}


def hr_agent(state: EnterpriseState) -> dict:
    return _enterprise_rag_agent("hr", "Recursos Humanos", state)


def it_agent(state: EnterpriseState) -> dict:
    return _enterprise_rag_agent("it", "Tecnología", state)


def finance_agent(state: EnterpriseState) -> dict:
    return _enterprise_rag_agent("finance", "Finanzas", state)


def legal_agent(state: EnterpriseState) -> dict:
    return _enterprise_rag_agent("legal", "Legal", state)


def fallback_agent(state: EnterpriseState) -> dict:
    return {
        "agent_response": (
            "Tu consulta no pudo ser asignada a un departamento específico. "
            "Por favor contactá directamente a: hr@empresa.com (RRHH), it@empresa.com (IT), "
            "finance@empresa.com (Finanzas) o legal@empresa.com (Legal)."
        )
    }


def enterprise_router(state: EnterpriseState) -> str:
    return {
        "hr":      "hr_agent",
        "it":      "it_agent",
        "finance": "finance_agent",
        "legal":   "legal_agent",
    }.get(state["department"], "fallback_agent")


print("Agentes especialistas definidos.")

In [ ]:
def evaluator_agent(state: EnterpriseState) -> dict:
    """
    LLM-as-a-judge: evalúa la respuesta del agente en 4 dimensiones (0-10 cada una).
    Devuelve scores numéricos y feedback textual.
    """
    eval_prompt = (
        "Sos un evaluador de calidad de respuestas de sistemas multiagente corporativos.\n"
        "Evaluá la siguiente respuesta en 4 dimensiones del 0 al 10:\n\n"
        "- relevance: ¿la respuesta es relevante a la consulta del empleado?\n"
        "- completeness: ¿la respuesta cubre todos los aspectos de la consulta?\n"
        "- accuracy: ¿la información es precisa y no contiene errores?\n"
        "- clarity: ¿la respuesta es clara y fácil de entender?\n\n"
        "Respondé SOLO con JSON válido sin markdown:\n"
        "{\"relevance\": N, \"completeness\": N, \"accuracy\": N, \"clarity\": N, \"feedback\": \"...\"}"
        f"\n\nConsulta original: {state['query']}\n"
        f"Respuesta del agente: {state['agent_response']}"
    )

    try:
        response = llm.invoke(eval_prompt)
        text = response.content.strip()
        text = re.sub(r"```[\w]*\n?", "", text).strip()
        data = json.loads(text)
        relevance     = float(data.get("relevance", 5))
        completeness  = float(data.get("completeness", 5))
        accuracy      = float(data.get("accuracy", 5))
        clarity       = float(data.get("clarity", 5))
        feedback      = str(data.get("feedback", ""))
        score         = round((relevance + completeness + accuracy + clarity) / 4, 2)
    except Exception as e:
        relevance = completeness = accuracy = clarity = 5.0
        score = 5.0
        feedback = f"Error al evaluar: {e}"

    return {
        "eval_score":        score,
        "eval_relevance":    relevance,
        "eval_completeness": completeness,
        "eval_accuracy":     accuracy,
        "eval_clarity":      clarity,
        "eval_feedback":     feedback,
    }


print("Evaluador definido.")

## Sección 4 — Compilar el grafo

In [ ]:
graph = StateGraph(EnterpriseState)

graph.add_node("enterprise_router_node", enterprise_router_node)
graph.add_node("hr_agent",               hr_agent)
graph.add_node("it_agent",               it_agent)
graph.add_node("finance_agent",          finance_agent)
graph.add_node("legal_agent",            legal_agent)
graph.add_node("fallback_agent",         fallback_agent)
graph.add_node("evaluator_agent",        evaluator_agent)

graph.add_edge(START, "enterprise_router_node")
graph.add_conditional_edges(
    "enterprise_router_node",
    enterprise_router,
    {
        "hr_agent":      "hr_agent",
        "it_agent":      "it_agent",
        "finance_agent": "finance_agent",
        "legal_agent":   "legal_agent",
        "fallback_agent": "fallback_agent",
    },
)
for node in ["hr_agent", "it_agent", "finance_agent", "legal_agent", "fallback_agent"]:
    graph.add_edge(node, "evaluator_agent")

graph.add_edge("evaluator_agent", END)

app = graph.compile()
print("Grafo compilado.")

## Demo — Consultas de empleados con evaluación automática

Cada consulta pasa por el especialista y luego por el evaluador. Los scores permiten monitorear la calidad del sistema.

In [ ]:
EMPTY = {
    "query": "", "department": "", "reason": "", "agent_response": "",
    "eval_score": 0.0, "eval_relevance": 0.0, "eval_completeness": 0.0,
    "eval_accuracy": 0.0, "eval_clarity": 0.0, "eval_feedback": "",
}

queries = [
    "¿Cuántos días de vacaciones tengo si llevo 6 años en la empresa?",
    "No puedo conectarme a la VPN desde mi casa",
    "Necesito reembolso de un gasto de $80.000 de un viaje de trabajo",
    "Un proveedor me pide firmar un contrato sin revisión legal, ¿puedo hacerlo?",
    "¿Dónde puedo conseguir café gratis en la oficina?",
]

for q in queries:
    trace = None
    if LANGFUSE_ENABLED and langfuse_client:
        trace = langfuse_client.trace(name="enterprise-query", input={"query": q})

    r = app.invoke({**EMPTY, "query": q})

    if trace:
        trace.update(output={"department": r["department"], "eval_score": r["eval_score"]})

    print(f"\nConsulta:    {q}")
    print(f"Departamento: {r['department']}")
    print(f"Respuesta:   {r['agent_response'][:100]}...")
    print(f"Score total: {r['eval_score']:.1f}/10  "
          f"(rel={r['eval_relevance']:.0f} comp={r['eval_completeness']:.0f} "
          f"acc={r['eval_accuracy']:.0f} clar={r['eval_clarity']:.0f})")
    print(f"Feedback:    {r['eval_feedback'][:80]}")
    print("-" * 70)

In [ ]:
print(app.get_graph().draw_mermaid())

## Checks automáticos

In [ ]:
def run_checks():
    empty = {
        "query": "", "department": "", "reason": "", "agent_response": "",
        "eval_score": 0.0, "eval_relevance": 0.0, "eval_completeness": 0.0,
        "eval_accuracy": 0.0, "eval_clarity": 0.0, "eval_feedback": "",
    }

    r1 = app.invoke({**empty, "query": "¿cuántos días de capacitación pagada tengo?"})
    assert r1["department"] == "hr", f"esperaba hr: {r1['department']}"
    assert isinstance(r1["eval_score"], float) and r1["eval_score"] >= 0
    assert isinstance(r1["eval_feedback"], str) and len(r1["eval_feedback"]) > 0

    r2 = app.invoke({**empty, "query": "¿qué herramientas SaaS están aprobadas para usar?"})
    assert r2["department"] == "it", f"esperaba it: {r2['department']}"

    r3 = app.invoke({**empty, "query": "¿cómo proceso el reembolso de un viaje corporativo?"})
    assert r3["department"] == "finance", f"esperaba finance: {r3['department']}"

    r4 = app.invoke({**empty, "query": "un proveedor quiere que firme un NDA esta semana"})
    assert r4["department"] == "legal", f"esperaba legal: {r4['department']}"

    # verificar que el evaluador siempre corre (incluso en fallback)
    r5 = app.invoke({**empty, "query": "¿cuál es el mejor restaurante cerca?"})
    assert r5["eval_score"] >= 0, "evaluador no corrió"

    print("Checks E19 OK")

run_checks()

## ¿Qué viste en este caso?

- Un sistema empresarial completo puede modelarse como un grafo con múltiples nodos especializados conectados secuencialmente.
- El patrón **LLM-as-a-judge** permite auditar automáticamente la calidad de las respuestas sin intervención humana.
- **Langfuse** agrega observabilidad de producción: cada consulta queda registrada con inputs, outputs y métricas.

| Concepto | Implementación en E19 |
|---|---|
| 4 departamentos + fallback | 5 agentes RAG en nodos separados |
| Evaluador automático | `evaluator_agent` → LLM devuelve JSON con 4 scores |
| LLM-as-a-judge | El mismo LLM que responde también evalúa (roles distintos) |
| Langfuse (opcional) | Trace por consulta, habilitado con `LANGFUSE_ENABLED=True` |
| Flujo post-agente | Todos los agentes → `evaluator_agent` → `END` |

## Próximo ejercicio

En **E20** vas a ver un sistema de programación con 5 tecnologías especializadas (React, Angular, NestJS, Express, Python) y un router que detecta área y tecnología en una sola clasificación.